In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [ ]:
def lookup_clusters_for_dates(
    clusters_df: pd.DataFrame,
    esm_run: str,
    dates: list
) -> pd.Series:
    """
    Given a DataFrame `clusters_df` with columns
      ['esm_run', 'time', 'cluster'],
    return a Series mapping each date in `dates` to its cluster.
    """
    # ensure datetime
    df = clusters_df.copy()
    df["time"] = pd.to_datetime(df["time"])
    df["date"] = df["time"].dt.date

    # filter
    df2 = df[df["esm_run"] == esm_run]
    out = {}
    for d in dates:
        pts = df2[df2["date"] == d]
        if not pts.empty:
            # if multiple times per day, you can take mode() or first()
            out[d] = int(pts["cluster"].mode()[0])
        else:
            out[d] = np.nan
    return pd.Series(out, name="cluster")

In [ ]:


def plot_psl_clusters_and_occurrences(
    cluster_centers,    # np.ndarray, shape (n_clusters, nlat, nlon)
    labels,             # np.ndarray, shape (ntime,), ints 0..n_clusters-1
    times,              # pd.DatetimeIndex or array of datetimes, length ntime
    lons, lats,         # 1d arrays for plotting
    lee_dates,          # list of datetime.date for LEE events
    output_path=".",
    vmin=-2.5, vmax=2.5
):
    """
    Create a 3×5 grid:
      Row 1: mean PSL anomalies per cluster (5 maps)
      Row 2a: relative cluster occurrence per day-of-year
      Row 2b: relative cluster occurrence per year
      Row 3: cluster maps on LEE‐event dates (up to 3 panels)
    """
    n_clusters = cluster_centers.shape[0]
    # Prepare occurrence tables --------------------------------------------
    df = pd.DataFrame({
        "time": pd.to_datetime(times),
        "cluster": labels
    })
    df.set_index("time", inplace=True)
    df["doy"] = df.index.dayofyear
    df["year"] = df.index.year

    # daily relative occurrence
    daily = (df.groupby(["doy", "cluster"])
               .size()
               .unstack(fill_value=0))
    daily = daily.div(daily.sum(axis=1), axis=0)

    # annual relative occurrence
    annual = (df.groupby(["year", "cluster"])
                .size()
                .unstack(fill_value=0))
    annual = annual.div(annual.sum(axis=1), axis=0)

    # LEE event cluster‐on‐map data ----------------------------------------
    # pick up to 3 unique dates in lee_dates present in df
    lee_df = pd.DataFrame({"date": pd.to_datetime(lee_dates)})
    lee_df["date"] = lee_df["date"].dt.date
    # join to get clusters
    df_dates = df.copy()
    df_dates["date"] = df_dates.index.date
    merged = df_dates.reset_index().merge(
        lee_df, on="date", how="inner"
    ).drop_duplicates("date").set_index("date")
    # For each date, pick cluster label
    lee_clusters = merged["cluster"].to_dict()  # date → cluster

    # Figure setup ---------------------------------------------------------
    fig = plt.figure(figsize=(15, 10))
    gs = fig.add_gridspec(3, 5, height_ratios=[1, 0.7, 0.7], hspace=0.3)

    # Row 1: cluster‐mean maps ---------------------------------------------
    for k in range(n_clusters):
        ax = fig.add_subplot(gs[0, k], projection=ccrs.PlateCarree())
        ax.set_title(f"Cluster {k+1}", fontsize=12, fontweight="bold")
        im = ax.contourf(
            lons, lats, cluster_centers[k],
            levels=np.linspace(vmin, vmax, 21),
            transform=ccrs.PlateCarree(),
            cmap="coolwarm", vmin=vmin, vmax=vmax
        )
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS)
        ax.set_aspect("auto")
        if k == 0:
            ax.text(-0.2, 1.05, "(a)", transform=ax.transAxes,
                    fontsize=14, fontweight="bold")
    # shared colorbar:
    cax = fig.add_axes([0.92, 0.65, 0.015, 0.25])
    cb = fig.colorbar(im, cax=cax, orientation="vertical")
    cb.set_label("PSL Anomaly (hPa)", fontsize=10)

    # Row 2a: day‐of‐year occurrence ---------------------------------------
    ax2 = fig.add_subplot(gs[1, :3])
    daily.plot(
        kind="bar", stacked=True, width=1.0, ax=ax2,
        legend=False
    )
    ax2.set_xlabel("Day of Year")
    ax2.set_ylabel("Rel. Occurrence")
    ax2.text(-0.05, 1.05, "(b)", transform=ax2.transAxes,
             fontsize=14, fontweight="bold")

    # Row 2b: annual occurrence -------------------------------------------
    ax3 = fig.add_subplot(gs[1, 3:])
    annual.plot(
        kind="bar", stacked=True, width=1.0, ax=ax3,
        legend=False
    )
    ax3.set_xlabel("Year")
    ax3.set_ylabel("")  # shared scale
    ax3.text(-0.05, 1.05, "(c)", transform=ax3.transAxes,
             fontsize=14, fontweight="bold")

    # shared legend beneath row 2
    handles, labels_ = ax2.get_legend_handles_labels()
    legend_labels = [f"CL{int(l)+1}" for l in labels_]
    fig.legend(
        handles, legend_labels,
        loc="lower center", ncol=n_clusters,
        bbox_to_anchor=(0.5, 0.35)
    )

    # Row 3: LEE‐event cluster maps ----------------------------------------
    for i, (date, cl) in enumerate(lee_clusters.items()):
        if i >= 3:
            break
        ax = fig.add_subplot(gs[2, i], projection=ccrs.PlateCarree())
        ax.set_title(f"LEE on {date}\nCluster {cl+1}")
        data = cluster_centers[cl]  # or extract the anomaly at that date
        im2 = ax.contourf(
            lons, lats, data,
            levels=np.linspace(vmin, vmax, 21),
            transform=ccrs.PlateCarree(),
            cmap="coolwarm"
        )
        ax.coastlines()
        ax.add_feature(cfeature.BORDERS)
        ax.set_aspect("auto")
        if i == 0:
            ax.text(-0.2, 1.05, "(d)", transform=ax.transAxes,
                    fontsize=14, fontweight="bold")

    # save & finish --------------------------------------------------------
    os.makedirs(output_path, exist_ok=True)
    png = os.path.join(output_path, "psl_clusters_occurrences_LEE.png")
    fig.savefig(png, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {png}")


In [ ]:

# -----------------------
# Example usage:
# -----------------------
plot_psl_clusters_and_occurrences(
    cluster_centers=your_centers,
    labels=your_labels,
    times=your_time_index,
    lons=your_lons,
    lats=your_lats,
    lee_dates=list_of_your_LEE_dates,
    output_path="results/plots"
)

df = pd.read_csv("path/to/your/clusters.csv")
clusters_on_LEE = lookup_clusters_for_dates(
    clusters_df=df,
    esm_run="ESM1",
    dates=[pd.Timestamp("2020-07-15").date(), ...]
)
print(clusters_on_LEE)